# 🎯 CV Generation Fine-Tuning — Final Version
## Dataset: Resume.csv (2484 resumes · 24 categories)
## Models: Qwen2.5 · DeepSeek · Mistral

## ✅ Cell 0 — Check GPU

In [ ]:
import torch, os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

assert torch.cuda.is_available(), """
❌ GPU not available!
Runtime → Change runtime type → T4 GPU → Save
Runtime → Disconnect and delete runtime → Run again
"""
print(f"✅ GPU : {torch.cuda.get_device_name(0)}")
print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

✅ GPU : Tesla T4
✅ VRAM: 15.6 GB


## 📦 Cell 1 — Install (run once → Restart session)

In [ ]:
!pip install -q \
    "trl==1.3.0" \
    "transformers==4.57.6" \
    "accelerate==1.13.0" \
    "bitsandbytes==0.45.5" \
    "peft==0.19.1" \
    "datasets>=4.7.0" \
    pandas scikit-learn rouge-score huggingface_hub sentencepiece

print("✅ Done — Runtime → Restart session → start from Cell 0")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.3 MB/s eta 0:00:00
✅ Done — Runtime → Restart session → start from Cell 0


In [ ]:
!pip install -q "torchao>=0.16.0"
print("✅ Done — Runtime → Restart session")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.2 MB/s eta 0:00:00
✅ Done — Runtime → Restart session


## 🔑 Cell 2 — Config & Login

In [ ]:
import os, gc, torch
from google.colab import userdata
from huggingface_hub import login

# ===== CONFIGURE =====
HF_TOKEN   = userdata.get('HF_TOKEN')       # set in Colab Secrets 🔑
DATA_PATH  = "/content/Resume.csv"          # upload Resume.csv here
OUTPUT_DIR = "/content/cv_models"
HF_REPO    = "your-username/cv-finetuned"   # ← change to your HF username
# =====================

os.makedirs(OUTPUT_DIR, exist_ok=True)
login(token=HF_TOKEN)

MODEL_REGISTRY = {
    "qwen25":   "Qwen/Qwen2.5-1.5B-Instruct",
    "deepseek": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    "mistral":  "mistralai/Mistral-7B-Instruct-v0.3",
    # Add after requesting access:
    # "gemma3":  "google/gemma-3-1b-it",
    # "llama3":  "meta-llama/Llama-3.2-1B-Instruct",
}

TARGET_MODULES = {
    "qwen25":   ["q_proj", "k_proj", "v_proj", "o_proj"],
    "deepseek": ["q_proj", "k_proj", "v_proj", "o_proj"],
    "mistral":  ["q_proj", "k_proj", "v_proj", "o_proj"],
    "gemma3":   ["q_proj", "k_proj", "v_proj", "o_proj"],
    "llama3":   ["q_proj", "k_proj", "v_proj", "o_proj"],
}

print("✅ Config ready")
print(f"Models: {list(MODEL_REGISTRY.keys())}")

✅ Config ready
Models: ['qwen25', 'deepseek', 'mistral']


## 📊 Cell 3 — Load Resume.csv & Build Training Data

In [ ]:
import pandas as pd
import numpy as np
import re
from datasets import Dataset
from sklearn.model_selection import train_test_split

df = pd.read_csv(DATA_PATH)
print(f"Shape    : {df.shape}")
print(f"Columns  : {df.columns.tolist()}")
print(f"Categories ({df['Category'].nunique()}):")
print(df['Category'].value_counts().to_string())

def clean_resume(text):
    text = re.sub(r'\s+', ' ', str(text))
    return text.strip()

def build_prompt(row):
    category = row['Category'].replace('-', ' ').title()
    profile  = clean_resume(row['Resume_str'])[:600]
    return (
        f"You are a professional CV writer. "
        f"Generate a complete, well-structured CV for a {category} professional.\n\n"
        f"Profile summary provided by user:\n{profile}\n\n"
        f"Generate a full professional CV:"
    )

def build_output(row):
    return clean_resume(row['Resume_str'])[:1200]

df = df.dropna(subset=['Resume_str', 'Category']).copy()
df['input_prompt'] = df.apply(build_prompt, axis=1)
df['output_cv']    = df.apply(build_output, axis=1)

train_df, val_df = train_test_split(
    df, test_size=0.1, random_state=42, stratify=df['Category']
)

train_dataset = Dataset.from_pandas(
    train_df[['input_prompt', 'output_cv']].reset_index(drop=True)
)
val_dataset = Dataset.from_pandas(
    val_df[['input_prompt', 'output_cv']].reset_index(drop=True)
)

print(f"\n✅ Train: {len(train_dataset)} | Val: {len(val_dataset)}")

Shape    : (2484, 4)
Columns  : ['ID', 'Resume_str', 'Resume_html', 'Category']
Categories (24):
Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ADVOCATE                  118
CHEF                      118
ENGINEERING               118
ACCOUNTANT                118
FINANCE                   118
FITNESS                   117
AVIATION                  117
SALES                     116
BANKING                   115
HEALTHCARE                115
CONSULTANT                115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22

✅ Train: 2235 | Val: 249


## 🔁 Cell 4 — Fine-Tune All Models

In [ ]:
import gc, torch, time
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType


def free_memory():
    gc.collect()
    torch.cuda.empty_cache()
    free = torch.cuda.mem_get_info()[0] / 1e9
    print(f"  Free VRAM: {free:.2f} GB")
    return free


def run_finetune(model_choice, train_ds, val_ds):
    print(f"\n{'='*60}")
    print(f"  🚀 {model_choice.upper()} — {MODEL_REGISTRY[model_choice]}")
    print(f"{'='*60}")
    t_start  = time.time()
    model_id  = MODEL_REGISTRY[model_choice]
    save_path = f"{OUTPUT_DIR}/{model_choice}"

    # ── Tokenizer ──────────────────────────────────────────
    tok = AutoTokenizer.from_pretrained(
        model_id, token=HF_TOKEN,
        trust_remote_code=True, use_fast=True
    )
    tok.pad_token    = tok.eos_token
    tok.padding_side = "right"

    # ── 4-bit QLoRA ────────────────────────────────────────
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    # ── Load model ─────────────────────────────────────────
    mdl = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb,
        device_map={"":  "cuda:0"},
        token=HF_TOKEN,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        dtype=torch.float16,           # ✅ dtype بدل torch_dtype
    )
    mdl.config.use_cache      = False
    mdl.config.pretraining_tp = 1
    mdl.enable_input_require_grads()   # ✅ بدل prepare_model_for_kbit_training

    # ── LoRA ───────────────────────────────────────────────
    lora = LoraConfig(
        r=8, lora_alpha=16,
        target_modules=TARGET_MODULES[model_choice],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    mdl = get_peft_model(mdl, lora)
    mdl.print_trainable_parameters()

    # ── Format + Tokenize ──────────────────────────────────
    def fmt(s):
        try:
            msgs = [
                {"role": "user",      "content": s['input_prompt']},
                {"role": "assistant", "content": s['output_cv']},
            ]
            text = tok.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=False
            )
        except Exception:
            text = f"{s['input_prompt']}\n\n{s['output_cv']}{tok.eos_token}"
        return {"text": text}

    def tokenize(s):
        return tok(
            s["text"],
            truncation=True,
            max_length=512,
            padding="max_length",
        )

    tr = train_ds.map(fmt,    remove_columns=train_ds.column_names)
    vl = val_ds.map(fmt,      remove_columns=val_ds.column_names)
    tr = tr.map(tokenize, batched=True, remove_columns=["text"])
    vl = vl.map(tokenize, batched=True, remove_columns=["text"])

    # ── Training ───────────────────────────────────────────
    args = TrainingArguments(
        output_dir=save_path,
        num_train_epochs=2,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=16,
        learning_rate=2e-4,
        warmup_steps=20,
        fp16=True,
        bf16=False,
        logging_steps=20,
        eval_strategy="steps",
        eval_steps=100,
        save_steps=200,
        save_total_limit=1,
        load_best_model_at_end=True,
        report_to="none",
        optim="paged_adamw_8bit",
        gradient_checkpointing=False,  # ✅ False — not compatible with 4-bit
        dataloader_num_workers=0,
        dataloader_pin_memory=False,
    )

    trainer = Trainer(
        model=mdl,
        args=args,
        train_dataset=tr,
        eval_dataset=vl,
        data_collator=DataCollatorForLanguageModeling(tok, mlm=False),
    )

    free_memory()
    train_result = trainer.train()
    train_loss   = round(train_result.training_loss, 4)
    train_time   = round((time.time() - t_start) / 60, 1)

    # ── Save ───────────────────────────────────────────────
    mdl.save_pretrained(save_path)
    tok.save_pretrained(save_path)
    print(f"  ✅ Saved: {save_path}")

    # ── Upload to HuggingFace ──────────────────────────────
    try:
        from huggingface_hub import HfApi
        HfApi().create_repo(
            repo_id=f"{HF_REPO}-{model_choice}",
            token=HF_TOKEN, private=True, exist_ok=True
        )
        mdl.push_to_hub(f"{HF_REPO}-{model_choice}", token=HF_TOKEN)
        tok.push_to_hub(f"{HF_REPO}-{model_choice}", token=HF_TOKEN)
        print(f"  ✅ Uploaded: {HF_REPO}-{model_choice}")
    except Exception as e:
        print(f"  ⚠️  HF upload skipped: {e}")

    # ── Cleanup ────────────────────────────────────────────
    del mdl, tok, trainer, tr, vl
    free_memory()

    return {
        "model":      model_choice,
        "train_loss": train_loss,
        "train_min":  train_time,
    }


# ── Run all models ─────────────────────────────────────
MODELS_TO_TRAIN = ["qwen25", "deepseek", "mistral"]

train_results = []
for m in MODELS_TO_TRAIN:
    try:
        r = run_finetune(m, train_dataset, val_dataset)
        train_results.append(r)
        print(f"✅ {m} | loss={r['train_loss']} | {r['train_min']} min")
    except Exception as e:
        print(f"❌ {m} failed: {e}")
        train_results.append({"model": m, "train_loss": None, "train_min": None})
        free_memory()

print("\n🎉 Training complete!")


  🚀 QWEN25 — Qwen/Qwen2.5-1.5B-Instruct


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/2235 [00:00<?, ? examples/s]

Map:   0%|          | 0/249 [00:00<?, ? examples/s]

  Free VRAM: 7.77 GB


Step,Training Loss,Validation Loss
100,1.453900,1.476989
200,1.404100,1.449497


  ✅ Saved: /content/cv_models/mistral
  ⚠️  HF upload skipped: (Request ID: Root=1-69f620ad-2029c0087173f22f06479dea;72c06902-770f-44d2-b830-ea9e6c0f0040)

403 Forbidden: You don't have the rights to create a model under the namespace "your-username".
Cannot access content at: https://huggingface.co/api/repos/create.
Make sure your token has the correct permissions.
  Free VRAM: 12.91 GB
✅ mistral | loss=1.4995 | 132.6 min

🎉 Training complete!


In [ ]:
from huggingface_hub import HfApi, whoami

# اعرفي اسمك
info     = whoami(token=HF_TOKEN)
username = info['name']
print(f"✅ Username: {username}")   # ← {username} مش {basmalaalaa029}

api = HfApi()

# ارفعي الـ 3 موديلات
for model_choice in ["qwen25", "deepseek", "mistral"]:
    save_path = f"/content/cv_models/{model_choice}"
    repo_id   = f"{username}/cv-finetuned-{model_choice}"

    print(f"\nUploading {model_choice}...")

    api.create_repo(
        repo_id=repo_id,
        token=HF_TOKEN,
        private=True,
        exist_ok=True
    )

    api.upload_folder(
        folder_path=save_path,
        repo_id=repo_id,
        token=HF_TOKEN,
    )
    print(f"✅ {model_choice} → huggingface.co/{repo_id}")

print("\n🎉 All models uploaded!")

✅ Username: basmalaalaa029

Uploading qwen25...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  .../checkpoint-200/scaler.pt: 100%|##########| 1.38kB / 1.38kB            

  ...ckpoint-200/rng_state.pth: 100%|##########| 14.6kB / 14.6kB            

...

  ...adapter_model.safetensors:  58%|#####8    | 16.0MB / 27.3MB            

  ...eckpoint-200/optimizer.pt: 100%|##########| 14.1MB / 14.1MB            

  ...adapter_model.safetensors:  58%|#####8    | 16.0MB / 27.3MB            

No files have been modified since last commit. Skipping to prevent empty commit.


✅ mistral → huggingface.co/basmalaalaa029/cv-finetuned-mistral

🎉 All models uploaded!


In [ ]:
from huggingface_hub import snapshot_download, whoami

info     = whoami(token=HF_TOKEN)
username = info['name']

# حملي الموديلات المحفوظة
for model_choice in ["qwen25", "deepseek", "mistral"]:
    repo_id   = f"{username}/cv-finetuned-{model_choice}"
    save_path = f"/content/cv_models/{model_choice}"

    print(f"Downloading {model_choice}...")
    snapshot_download(
        repo_id=repo_id,
        local_dir=save_path,
        token=HF_TOKEN,
    )
    print(f"✅ {model_choice} ready")

# النتائج اللي عندك من التدريب
train_results = [
    {"model": "qwen25",   "train_loss": 1.8171, "train_min": 29.0},
    {"model": "deepseek", "train_loss": 2.4420, "train_min": 31.8},
    {"model": "mistral",  "train_loss": 1.4995, "train_min": 132.6},
]

MODELS_TO_TRAIN = ["qwen25", "deepseek", "mistral"]
print("\n✅ Ready to evaluate!")

Fetching 27 files:   0%|          | 0/27 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

trainer_state.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/437 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

✅ mistral ready

✅ Ready to evaluate!


## 📊 Cell 5 — Evaluate All Models (ROUGE + Speed)

In [ ]:
import json, time
import numpy as np
from rouge_score import rouge_scorer as rouge_lib
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

test_samples = val_df.head(15).to_dict('records')


def generate_cv_eval(row, mdl, tok):
    category = row['Category'].replace('-', ' ').title()
    profile  = clean_resume(row['Resume_str'])[:400]
    prompt   = (
        f"You are a professional CV writer. "
        f"Generate a complete CV for a {category} professional.\n\n"
        f"Profile:\n{profile}\n\nCV:"
    )
    inputs = tok(
        prompt, return_tensors="pt",
        truncation=True, max_length=300
    ).to(mdl.device)
    t0 = time.time()
    with torch.no_grad():
        out = mdl.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.7,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tok.eos_token_id,
        )
    elapsed = time.time() - t0
    gen_ids = out[0][inputs['input_ids'].shape[1]:]
    return tok.decode(gen_ids, skip_special_tokens=True), elapsed


def evaluate_model(model_choice, samples):
    print(f"\nEvaluating {model_choice.upper()}...")
    save_path = f"{OUTPUT_DIR}/{model_choice}"

    tok  = AutoTokenizer.from_pretrained(save_path, trust_remote_code=True)
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_REGISTRY[model_choice],
        token=HF_TOKEN,
        dtype=torch.float16,
        device_map={"": "cuda:0"},
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
    mdl = PeftModel.from_pretrained(base, save_path)
    mdl.eval()

    scorer = rouge_lib.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'], use_stemmer=True
    )
    scores, times = [], []

    for i, s in enumerate(samples):
        ref          = clean_resume(s['Resume_str'])[:1200]
        gen, elapsed = generate_cv_eval(s, mdl, tok)
        sc           = scorer.score(ref, gen)
        scores.append({
            'rouge1': sc['rouge1'].fmeasure,
            'rouge2': sc['rouge2'].fmeasure,
            'rougeL': sc['rougeL'].fmeasure,
        })
        times.append(elapsed)
        print(f"  [{i+1}/{len(samples)}] ROUGE-L={sc['rougeL'].fmeasure:.3f} | {elapsed:.1f}s")

    result = {
        "model":   model_choice,
        "rouge1":  round(float(np.mean([s['rouge1'] for s in scores])), 3),
        "rouge2":  round(float(np.mean([s['rouge2'] for s in scores])), 3),
        "rougeL":  round(float(np.mean([s['rougeL'] for s in scores])), 3),
        "avg_sec": round(float(np.mean(times)), 1),
    }
    print(f"  → Avg ROUGE-L: {result['rougeL']} | Speed: {result['avg_sec']}s")

    del mdl, base, tok
    free_memory()
    return result


# ── Run evaluation ─────────────────────────────────────
eval_results = []
for m in MODELS_TO_TRAIN:
    try:
        r = evaluate_model(m, test_samples)
        eval_results.append(r)
    except Exception as e:
        print(f"  ❌ {m} eval failed: {e}")
        free_memory()

with open(f"{OUTPUT_DIR}/comparison.json", "w") as f:
    json.dump(eval_results, f, indent=2)
print("\n💾 Results saved")


Evaluating QWEN25...
  [1/15] ROUGE-L=0.371 | 22.9s
  [2/15] ROUGE-L=0.359 | 21.5s
  [3/15] ROUGE-L=0.158 | 21.3s
  [4/15] ROUGE-L=0.359 | 20.7s
  [5/15] ROUGE-L=0.360 | 21.0s
  [6/15] ROUGE-L=0.149 | 21.4s
  [7/15] ROUGE-L=0.364 | 21.2s
  [8/15] ROUGE-L=0.127 | 20.8s
  [9/15] ROUGE-L=0.171 | 21.2s
  [10/15] ROUGE-L=0.116 | 21.4s
  [11/15] ROUGE-L=0.204 | 20.4s
  [12/15] ROUGE-L=0.403 | 21.1s
  [13/15] ROUGE-L=0.365 | 21.4s
  [14/15] ROUGE-L=0.123 | 21.1s
  [15/15] ROUGE-L=0.361 | 21.1s
  → Avg ROUGE-L: 0.266 | Speed: 21.2s
  Free VRAM: 15.47 GB

Evaluating DEEPSEEK...
  [1/15] ROUGE-L=0.161 | 27.5s
  [2/15] ROUGE-L=0.296 | 21.3s
  [3/15] ROUGE-L=0.346 | 27.7s
  [4/15] ROUGE-L=0.326 | 23.1s
  [5/15] ROUGE-L=0.290 | 21.8s
  [6/15] ROUGE-L=0.342 | 21.9s
  [7/15] ROUGE-L=0.119 | 26.5s
  [8/15] ROUGE-L=0.376 | 21.9s
  [9/15] ROUGE-L=0.105 | 21.4s
  [10/15] ROUGE-L=0.115 | 21.4s
  [11/15] ROUGE-L=0.200 | 21.8s
  [12/15] ROUGE-L=0.340 | 21.9s
  [13/15] ROUGE-L=0.165 | 21.6s
  [14/15] ROUGE-

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

  [1/15] ROUGE-L=0.390 | 24.6s
  [2/15] ROUGE-L=0.360 | 24.3s
  [3/15] ROUGE-L=0.494 | 24.4s
  [4/15] ROUGE-L=0.354 | 24.3s
  [5/15] ROUGE-L=0.389 | 24.4s
  [6/15] ROUGE-L=0.443 | 29.4s
  [7/15] ROUGE-L=0.419 | 24.6s
  [8/15] ROUGE-L=0.475 | 25.0s
  [9/15] ROUGE-L=0.424 | 25.2s
  [10/15] ROUGE-L=0.399 | 25.2s
  [11/15] ROUGE-L=0.431 | 28.9s
  [12/15] ROUGE-L=0.398 | 25.8s
  [13/15] ROUGE-L=0.376 | 25.3s
  [14/15] ROUGE-L=0.416 | 25.9s
  [15/15] ROUGE-L=0.458 | 25.5s
  → Avg ROUGE-L: 0.415 | Speed: 25.5s
  Free VRAM: 15.47 GB

💾 Results saved


## 🏆 Cell 6 — Comparison Table & Best Model

In [ ]:
import pandas as pd

loss_map = {r['model']: r for r in train_results}
for r in eval_results:
    tr = loss_map.get(r['model'], {})
    r['train_loss'] = tr.get('train_loss')
    r['train_min']  = tr.get('train_min')

results_df = pd.DataFrame(eval_results).sort_values('rougeL', ascending=False)
results_df = results_df.reset_index(drop=True)
results_df.index += 1

print("\n" + "="*70)
print("              📊 MODEL COMPARISON — Resume.csv")
print("="*70)
print(results_df[[
    'model', 'rouge1', 'rouge2', 'rougeL', 'avg_sec', 'train_loss', 'train_min'
]].to_string())
print("="*70)
print()
print("Metric guide:")
print("  ROUGE-L    → quality of generated CV  (higher = better)")
print("  avg_sec    → seconds to generate 1 CV (lower  = better)")
print("  train_loss → how well model learned   (lower  = better)")
print()

best = results_df.iloc[0]
print(f"🏆 Best Model : {best['model'].upper()}")
print(f"   ROUGE-L   : {best['rougeL']}")
print(f"   ROUGE-1   : {best['rouge1']}")
print(f"   Speed     : {best['avg_sec']}s per CV")
print(f"   Train Loss: {best['train_loss']}")
print(f"\n✅ Use '{best['model']}' in your app!")

BEST_MODEL = best['model']


              📊 MODEL COMPARISON — Resume.csv
      model  rouge1  rouge2  rougeL  avg_sec  train_loss  train_min
1   mistral   0.502   0.340   0.415     25.5      1.4995      132.6
2    qwen25   0.382   0.181   0.266     21.2      1.8171       29.0
3  deepseek   0.312   0.151   0.218     22.9      2.4420       31.8

Metric guide:
  ROUGE-L    → quality of generated CV  (higher = better)
  avg_sec    → seconds to generate 1 CV (lower  = better)
  train_loss → how well model learned   (lower  = better)

🏆 Best Model : MISTRAL
   ROUGE-L   : 0.415
   ROUGE-1   : 0.502
   Speed     : 25.5s per CV
   Train Loss: 1.4995

✅ Use 'mistral' in your app!


## 🧪 Cell 7 — Test Best Model: Generate CV from User Profile

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

save_path = f"{OUTPUT_DIR}/{BEST_MODEL}"
tok  = AutoTokenizer.from_pretrained(save_path, trust_remote_code=True)
base = AutoModelForCausalLM.from_pretrained(
    MODEL_REGISTRY[BEST_MODEL],
    token=HF_TOKEN,
    dtype=torch.float16,               # ✅ dtype بدل torch_dtype
    device_map={"":  "cuda:0"},
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
mdl = PeftModel.from_pretrained(base, save_path)
mdl.eval()
print(f"✅ Loaded: {BEST_MODEL.upper()}")


def generate_cv(user_profile: dict) -> str:
    """
    Main app function — called when user clicks 'Generate CV'.
    Input : user profile dict
    Output: generated CV string
    """
    category = user_profile.get('job_category', 'Professional')
    prompt   = (
        f"You are a professional CV writer. "
        f"Generate a complete CV for a {category} professional.\n\n"
        f"Name      : {user_profile.get('name', '')}\n"
        f"Email     : {user_profile.get('email', '')}\n"
        f"Phone     : {user_profile.get('phone', '')}\n"
        f"Skills    : {user_profile.get('skills', '')}\n"
        f"Experience: {user_profile.get('experience', '')}\n"
        f"Education : {user_profile.get('education', '')}\n"
        f"Summary   : {user_profile.get('summary', '')}\n\n"
        f"Generate a full professional CV:"
    )
    inputs = tok(
        prompt, return_tensors="pt",
        truncation=True, max_length=350
    ).to(mdl.device)
    with torch.no_grad():
        out = mdl.generate(
            **inputs,
            max_new_tokens=500,
            temperature=0.7,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tok.eos_token_id,
        )
    gen_ids = out[0][inputs['input_ids'].shape[1]:]
    return tok.decode(gen_ids, skip_special_tokens=True)


# ── Test ───────────────────────────────────────────────
user_profile = {
    "name":         "basmala alaa ahmed",
    "email":        "basmalaalaa029@email.com",
    "phone":        "+20 100 123 4567",
    "job_category": "AI Enginer",
    "skills":       "Python, Machine Learning, TensorFlow, FastAPI, SQL, Docker, Git",
    "experience":   "ML Engineer at Tech Corp 2021-2024. Junior Developer at Startup 2019-2021.",
    "education":    "B.Sc. Computer Science, Cairo University, 2019. GPA 3.8/4.0",
    "summary":      "Passionate ML engineer with 4+ years of production experience.Passionate ML engineer with 4+ years of production experience ",
}

print("\nGenerating CV...")
cv = generate_cv(user_profile)
print("\n" + "="*60)
print("GENERATED CV:")
print("="*60)
print(cv)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 1.81 MiB is free. Including non-PyTorch memory, this process has 14.56 GiB memory in use. Of the allocated memory 14.29 GiB is allocated by PyTorch, and 144.00 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
!pip install -q reportlab

import re
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer,
    HRFlowable, Table, TableStyle
)
from reportlab.lib.enums import TA_LEFT, TA_CENTER


def clean_cv_text(text):
    """Remove # symbols and clean up generated text."""
    text = re.sub(r'#+', '', text)          # ازيل الـ #
    text = re.sub(r'\*+', '', text)         # ازيل الـ *
    text = re.sub(r'\s+', ' ', text)        # spaces متعددة
    return text.strip()


def generate_cv_pdf(cv_text, user_profile, output_path="/content/CV_Generated.pdf"):

    doc = SimpleDocTemplate(
        output_path, pagesize=A4,
        leftMargin=2*cm, rightMargin=2*cm,
        topMargin=2*cm,  bottomMargin=2*cm,
    )

    DARK_BLUE  = colors.HexColor("#1a2e4a")
    MID_BLUE   = colors.HexColor("#2c5282")
    TEXT_COLOR = colors.HexColor("#2d2d2d")

    name_style = ParagraphStyle(
        "Name", fontName="Helvetica-Bold", fontSize=24,
        textColor=colors.white, alignment=TA_CENTER, spaceAfter=4,
    )
    title_style = ParagraphStyle(
        "Title", fontName="Helvetica", fontSize=11,
        textColor=colors.HexColor("#cce0ff"), alignment=TA_CENTER, spaceAfter=4,
    )
    contact_style = ParagraphStyle(
        "Contact", fontName="Helvetica", fontSize=9,
        textColor=colors.white, alignment=TA_CENTER,
    )
    section_style = ParagraphStyle(
        "Section", fontName="Helvetica-Bold", fontSize=11,
        textColor=MID_BLUE, spaceBefore=14, spaceAfter=4,
    )
    body_style = ParagraphStyle(
        "Body", fontName="Helvetica", fontSize=9.5,
        textColor=TEXT_COLOR, leading=14, spaceAfter=4,
    )
    bullet_style = ParagraphStyle(
        "Bullet", fontName="Helvetica", fontSize=9.5,
        textColor=TEXT_COLOR, leading=14, leftIndent=12, spaceAfter=2,
    )

    story = []

    # ── Header ─────────────────────────────────────────
    header_table = Table(
        [[Paragraph(user_profile.get("name", ""), name_style)]],
        colWidths=[17*cm]
    )
    header_table.setStyle(TableStyle([
        ("BACKGROUND",   (0,0), (-1,-1), DARK_BLUE),
        ("TOPPADDING",   (0,0), (-1,-1), 18),
        ("BOTTOMPADDING",(0,0), (-1,-1), 6),
    ]))
    story.append(header_table)

    title_table = Table(
        [[Paragraph(user_profile.get("job_category", ""), title_style)]],
        colWidths=[17*cm]
    )
    title_table.setStyle(TableStyle([
        ("BACKGROUND",   (0,0), (-1,-1), DARK_BLUE),
        ("TOPPADDING",   (0,0), (-1,-1), 0),
        ("BOTTOMPADDING",(0,0), (-1,-1), 6),
    ]))
    story.append(title_table)

    contact_text = f"{user_profile.get('email','')}   |   {user_profile.get('phone','')}"
    contact_table = Table(
        [[Paragraph(contact_text, contact_style)]],
        colWidths=[17*cm]
    )
    contact_table.setStyle(TableStyle([
        ("BACKGROUND",   (0,0), (-1,-1), MID_BLUE),
        ("TOPPADDING",   (0,0), (-1,-1), 6),
        ("BOTTOMPADDING",(0,0), (-1,-1), 8),
    ]))
    story.append(contact_table)
    story.append(Spacer(1, 10))

    def section_header(title):
        story.append(Paragraph(title.upper(), section_style))
        story.append(HRFlowable(
            width="100%", thickness=1.5,
            color=MID_BLUE, spaceAfter=6
        ))

    # ── Summary — من الـ user_profile مش من الـ generated ──
    summary = clean_cv_text(user_profile.get("summary", ""))
    if summary:
        section_header("Professional Summary")
        story.append(Paragraph(summary, body_style))

    # ── Skills ─────────────────────────────────────────
    skills_raw = user_profile.get("skills", "")
    if skills_raw:
        section_header("Core Skills")
        skill_list = [s.strip() for s in skills_raw.split(",") if s.strip()]
        rows = []
        row  = []
        for skill in skill_list:
            row.append(Paragraph(f"• {skill}", bullet_style))
            if len(row) == 3:
                rows.append(row); row = []
        if row:
            while len(row) < 3:
                row.append(Paragraph("", body_style))
            rows.append(row)
        if rows:
            skill_table = Table(rows, colWidths=[5.5*cm, 5.5*cm, 5.5*cm])
            skill_table.setStyle(TableStyle([
                ("VALIGN",       (0,0), (-1,-1), "TOP"),
                ("TOPPADDING",   (0,0), (-1,-1), 2),
                ("BOTTOMPADDING",(0,0), (-1,-1), 2),
            ]))
            story.append(skill_table)

    # ── Experience ─────────────────────────────────────
    exp_raw = user_profile.get("experience", "")
    if exp_raw:
        section_header("Work Experience")
        # Split on . أو newline
        for part in re.split(r'\.\s+(?=[A-Z])', exp_raw):
            part = part.strip().rstrip('.')
            if part:
                story.append(Paragraph(f"• {part}", bullet_style))
                story.append(Spacer(1, 3))

    # ── Education ──────────────────────────────────────
    edu_raw = user_profile.get("education", "")
    if edu_raw:
        section_header("Education")
        # Split على . بس لو اللي بعدها حرف كبير أو نهاية النص
        parts = re.split(r'\.\s+(?=[A-Z])', edu_raw)
        for part in parts:
            part = part.strip().rstrip('.')
            if part:
                story.append(Paragraph(f"• {part}", bullet_style))
                story.append(Spacer(1, 3))

    # ── Generated CV content (cleaned) ─────────────────
    section_header("Additional Details")
    cleaned_cv = clean_cv_text(cv_text)
    # شيلي أول سطر لو فيه كلام زي "Generate a CV"
    lines = [l.strip() for l in cleaned_cv.split('\n') if l.strip()]
    for line in lines:
        if any(skip in line.lower() for skip in [
            "generate a", "you are", "professional cv writer",
            "user profile", "full professional"
        ]):
            continue
        story.append(Paragraph(f"• {line}", bullet_style))

    # ── Footer ─────────────────────────────────────────
    story.append(Spacer(1, 20))
    story.append(HRFlowable(width="100%", thickness=0.5, color=colors.lightgrey))
    story.append(Paragraph(
        "Generated by CV Generator AI",
        ParagraphStyle("Footer", fontName="Helvetica",
                       fontSize=8, textColor=colors.grey, alignment=TA_CENTER)
    ))

    doc.build(story)
    print(f"✅ PDF saved: {output_path}")
    return output_path


# ── Generate ───────────────────────────────────────────
pdf_path = generate_cv_pdf(cv, user_profile)

# ── Download ───────────────────────────────────────────
from google.colab import files
files.download(pdf_path)
print("📥 CV PDF downloaded!")

✅ PDF saved: /content/CV_Generated.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 CV PDF downloaded!


## 🌐 Cell 8 — FastAPI Server

In [ ]:
api_code = f'''
# Run: tmux new -s api → uvicorn api_server:app --host 0.0.0.0 --port 8000

from fastapi import FastAPI
from pydantic import BaseModel
from typing import Optional
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

app       = FastAPI(title="CV Generator")
BEST      = "{BEST_MODEL}"
SAVE_PATH = f"./cv_models/{{BEST}}"
MODEL_ID  = "{MODEL_REGISTRY[BEST_MODEL]}"
HF_TOKEN  = "your_hf_token_here"

print(f"Loading {{BEST}} model...")
tok  = AutoTokenizer.from_pretrained(SAVE_PATH, trust_remote_code=True)
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.float16,
    device_map={{"":  "cuda:0"}},
    token=HF_TOKEN, trust_remote_code=True, low_cpu_mem_usage=True
)
model = PeftModel.from_pretrained(base, SAVE_PATH)
model.eval()
print("Model ready ✅")


class UserProfile(BaseModel):
    name:         str
    email:        str
    job_category: Optional[str] = "Professional"
    phone:        Optional[str] = ""
    skills:       Optional[str] = ""
    experience:   Optional[str] = ""
    education:    Optional[str] = ""
    summary:      Optional[str] = ""


@app.post("/generate-cv")
async def generate_cv_endpoint(p: UserProfile):
    prompt = (
        f"Generate a professional CV for a {{p.job_category}} professional.\\n\\n"
        f"Name: {{p.name}}\\nEmail: {{p.email}}\\nPhone: {{p.phone}}\\n"
        f"Skills: {{p.skills}}\\nExperience: {{p.experience}}\\n"
        f"Education: {{p.education}}\\nSummary: {{p.summary}}\\n\\nCV:"
    )
    inputs = tok(
        prompt, return_tensors="pt",
        truncation=True, max_length=350
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=500,
            temperature=0.7, do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tok.eos_token_id
        )
    gen = out[0][inputs["input_ids"].shape[1]:]
    return {{"cv": tok.decode(gen, skip_special_tokens=True), "model": BEST}}


@app.get("/health")
def health():
    return {{"status": "ok", "model": BEST}}
'''

with open("/content/api_server.py", "w") as f:
    f.write(api_code)

print("✅ api_server.py saved")
print()
print("Run in terminal:")
print("  tmux new -s api")
print("  pip install fastapi uvicorn")
print("  uvicorn api_server:app --host 0.0.0.0 --port 8000")
print()
print("Frontend POST to: http://your-server:8000/generate-cv")
print('  Body: { "name": "...", "email": "...", "job_category": "IT", "skills": "..." }')